[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week09.ipynb)

# Week 9: Convolutional Networks II
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Batch and layer normalization; residual connections; modern CNN design.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Add normalization and residual connections.
- Understand why these help deeper networks train.
- Measure the effect of each with an ablation.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. A residual block
out = F(x) + x keeps the input shape and adds a skip path.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1); self.bn1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1); self.bn2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.bn1(self.c1(x)))
        h = self.bn2(self.c2(h))
        return F.relu(h + x)              # the skip connection
print("ResBlock keeps shape:", tuple(ResBlock(16)(torch.randn(2, 16, 8, 8)).shape))

## 2. What batch norm does to activations
It re-centers and re-scales each feature; look at the histogram.

In [ ]:
x = torch.randn(1000, 1) * 4 + 7          # mean 7, std 4
bn = nn.BatchNorm1d(1); bn.train(); out = bn(x).detach()
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(x.squeeze().tolist(), bins=30); ax[0].set_title("before BN (mean 7)")
ax[1].hist(out.squeeze().tolist(), bins=30); ax[1].set_title("after BN (mean ~0, std ~1)")
plt.show()

## 3. Batch norm helps a deep net train
Ablation: with vs without batch normalization.

In [ ]:
torch.manual_seed(0)
X = torch.randn(256, 16); y = torch.randint(0, 2, (256,))
def deepnet(bn):
    layers = []
    for _ in range(6):
        layers.append(nn.Linear(16, 16))
        if bn: layers.append(nn.BatchNorm1d(16))
        layers.append(nn.ReLU())
    return nn.Sequential(*layers, nn.Linear(16, 2))
for bn in [False, True]:
    m = deepnet(bn); o = torch.optim.SGD(m.parameters(), 0.1); f = nn.CrossEntropyLoss(); h = []
    for _ in range(80):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label="with BN" if bn else "no BN")
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Normalization ablation"); plt.show()

## 4. Residual connections at depth
Plain 12-layer net vs the same with skips.

In [ ]:
class Deep(nn.Module):
    def __init__(self, res):
        super().__init__(); self.res = res
        self.blocks = nn.ModuleList([nn.Linear(16, 16) for _ in range(12)])
        self.head = nn.Linear(16, 2)
    def forward(self, x):
        for b in self.blocks:
            h = F.relu(b(x)); x = h + x if self.res else h
        return self.head(x)
for res in [False, True]:
    m = Deep(res); o = torch.optim.SGD(m.parameters(), 0.05); f = nn.CrossEntropyLoss(); h = []
    for _ in range(100):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label="residual" if res else "plain")
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Residual vs plain (12 layers)"); plt.show()

## 5. Skips let gradients reach the first layer
Measure the gradient norm at the first block: plain vs residual.

In [ ]:
def first_grad(res):
    m = Deep(res); f = nn.CrossEntropyLoss()
    f(m(X), y).backward()
    return m.blocks[0].weight.grad.norm().item()
print(f"first-layer grad norm  plain: {first_grad(False):.2e}")
print(f"first-layer grad norm  resid: {first_grad(True):.2e}  (larger = signal reaches the start)")

## 6. LayerNorm vs BatchNorm
LayerNorm normalizes per sample (batch-independent), used in sequence models.

In [ ]:
x = torch.randn(4, 8)
ln = nn.LayerNorm(8)(x)
print("LayerNorm: each ROW has mean ~0:", ln.mean(dim=1).round(decimals=2).tolist())
bn = nn.BatchNorm1d(8)(x)
print("BatchNorm: each COLUMN has mean ~0:", bn.mean(dim=0).round(decimals=2).tolist())

## 7. Batch norm: train vs eval statistics
It uses batch stats in training, running stats at eval.

In [ ]:
bn = nn.BatchNorm1d(4)
x = torch.randn(8, 4) * 5 + 3
bn.train(); _ = bn(x)
print("running mean after a train batch:", bn.running_mean.round(decimals=2).tolist())
bn.eval(); print("eval uses those running stats, not the current batch.")

### Mini-exercise
Build an 18-layer plain net and an 18-layer residual net (reuse `Deep`) and compare the final loss after 100 SGD steps. Does the gap widen with depth?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
class DeepN(nn.Module):
    def __init__(self, res, n=18):
        super().__init__(); self.res = res
        self.blocks = nn.ModuleList([nn.Linear(16, 16) for _ in range(n)]); self.head = nn.Linear(16, 2)
    def forward(self, x):
        for b in self.blocks:
            h = F.relu(b(x)); x = h + x if self.res else h
        return self.head(x)
for res in [False, True]:
    m = DeepN(res); o = torch.optim.SGD(m.parameters(), 0.05); f = nn.CrossEntropyLoss()
    for _ in range(100):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step()
    print(("residual" if res else "plain   "), "final loss", round(l.item(), 3))

**Recap:** normalization stabilizes and accelerates training; residual connections give gradients a direct path so very deep nets train; compare training curves, not just final accuracy.

---
Student materials for this week: the lab handout (`labs/week09.html`) and the curated references (`references/week09.html`) in the course site.